# TI3130: Data Wrangling &mdash; Exercises
**Julián Urbano &mdash; November 2025**

In [1]:
import sys
import numpy as np
import pandas as pd
print("python ", sys.version,
      "\nnumpy", np.__version__,
      "\npandas", pd.__version__)

python  3.12.4 | packaged by Anaconda, Inc. | (main, Jun 18 2024, 15:03:56) [MSC v.1929 64 bit (AMD64)] 
numpy 1.26.4 
pandas 2.2.2


For these exercises we will use the _AirBnB Berlin_ datasets. Please refer to their HTML files for a description of the variables.

In [3]:
listings = pd.read_csv("airbnb_listings.csv")
reviews = pd.read_csv("airbnb_reviews.csv")

**1) The number of distinct `host_id`s (4507) is different from the number of distinct `host_name`s (2118). Why is that? What are the implications?**

Several hostst can have the same name, however they all got an unique id.

**2) Create a new column called `minimum_price` that calculates the minimum price of a listing given its `minimum_nights` and `price`.**

In [32]:
listings['minimum_price'] = listings['minimum_nights'] * listings['price']

**3) What are the listings that have reviews not included in the `reviews` data frame? Tip: you can know if one variable `x` is `NaN` via `x != x`; this is because `NaN` equals to nothing, not even `NaN` itself.**

In [223]:
rcount = reviews.groupby(['listing_id'], as_index=False).aggregate(n = ('listing_id', 'count'))
lcount = listings[listings['number_of_reviews'] > 0]
merged = lcount.merge(rcount, left_on='id', right_on='listing_id', how='left')
df3 = merged[merged['listing_id'] != merged['listing_id']]
display(df3['id'])

704      773695
733      804227
734      804949
735      808174
736      810056
         ...   
4386    7491644
4387    7492748
4388    7495407
4389    7495977
4390    7496395
Name: id, Length: 3659, dtype: int64

**4) Create a data frame where rows are `host_ids`, columns are `neighborhood_group`s, and values are the mean `price` of the listings of that host in that neighborhood group. Round the price to full euros.**

In [172]:
aggregation = listings.groupby(['host_id', 'neighborhood_group'], as_index=False).aggregate(mean = ('price', 'mean'))
df4 = aggregation.pivot(index = 'host_id', columns = 'neighborhood_group', values = 'mean').round(0)

display(df4)

neighborhood_group,Charlottenburg-Wilm.,Friedrichshain-Kreuzberg,Lichtenberg,Marzahn - Hellersdorf,Mitte,Neukölln,Pankow,Reinickendorf,Spandau,Steglitz - Zehlendorf,Tempelhof - Schöneberg,Treptow - Köpenick
host_id,,,,,,,,,,,,
2217,NaN,NaN,NaN,NaN,60.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2986,NaN,NaN,NaN,NaN,NaN,NaN,17.0,NaN,NaN,NaN,NaN,NaN
3718,NaN,NaN,NaN,NaN,NaN,NaN,90.0,NaN,NaN,NaN,NaN,NaN
4108,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,26.0,NaN
11015,NaN,49.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
39219660,NaN,78.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
39222511,NaN,NaN,NaN,NaN,NaN,NaN,NaN,74.0,NaN,NaN,NaN,NaN
39230948,NaN,NaN,NaN,NaN,87.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**5) Same as 4), but show only cases where the average price is cheaper than 30€/night.**

In [194]:
df5 = df4[df4 < 30].dropna(how='all')
display(df5)

neighborhood_group,Charlottenburg-Wilm.,Friedrichshain-Kreuzberg,Lichtenberg,Marzahn - Hellersdorf,Mitte,Neukölln,Pankow,Reinickendorf,Spandau,Steglitz - Zehlendorf,Tempelhof - Schöneberg,Treptow - Köpenick
host_id,,,,,,,,,,,,
2986,NaN,NaN,NaN,NaN,NaN,NaN,17.0,NaN,NaN,NaN,NaN,NaN
4108,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,26.0,NaN
12723,NaN,NaN,NaN,NaN,NaN,NaN,16.0,NaN,NaN,NaN,NaN,NaN
130019,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,25.0,NaN
159734,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,25.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
39137222,NaN,NaN,NaN,NaN,25.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
39152545,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22.0,NaN
39156665,NaN,NaN,NaN,NaN,NaN,NaN,28.0,NaN,NaN,NaN,NaN,NaN


**6) Take the result of 4) and turn it into long format. Drop any rows with missing values.**

In [200]:
df6 = df4.reset_index().melt(id_vars = 'host_id', var_name = 'neighborhood_group', value_name = 'mean_price').dropna()
display(df6)

,host_id,neighborhood_group,mean_price
47,156897,Charlottenburg-Wilm.,46.0
73,221653,Charlottenburg-Wilm.,61.0
81,261488,Charlottenburg-Wilm.,84.0
84,264072,Charlottenburg-Wilm.,55.0
90,276540,Charlottenburg-Wilm.,70.0
...,...,...,...
53877,36314853,Treptow - Köpenick,50.0
53902,36762019,Treptow - Köpenick,50.0
53933,37375563,Treptow - Köpenick,65.0
53948,37553259,Treptow - Köpenick,20.0


**7) The following code snippets are two different solutions to the problem of selecting the top 10 neighborhoods by some price-related criterion. One uses [`head`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.head.html) and the other uses [`nlargest`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.nlargest.html). Why do they give different (number of) results?**

In [202]:
listings \
    .query('price > 200') \
    .groupby('neighborhood', as_index=False) \
    .aggregate(n = ('neighborhood', 'count')) \
    .sort_values('n', ascending=False) \
    .head(10)

,neighborhood,n
7,Helmholtzplatz,8
1,Brunnenstr. Süd,5
18,Regierungsviertel,4
15,Prenzlauer Berg Süd,4
23,Schöneberg-Nord,4
28,Tempelhofer Vorstadt,3
26,Südliche Friedrichstadt,3
24,Schöneberg-Süd,3
19,Reuterstraße,3
16,Prenzlauer Berg Südwest,3


In [204]:
listings \
    .query('price > 200') \
    .groupby('neighborhood', as_index=False) \
    .aggregate(n = ('neighborhood', 'count')) \
    .nlargest(n = 10, columns = 'n', keep = 'all')

,neighborhood,n
7,Helmholtzplatz,8
1,Brunnenstr. Süd,5
15,Prenzlauer Berg Süd,4
18,Regierungsviertel,4
23,Schöneberg-Nord,4
0,Alexanderplatz,3
16,Prenzlauer Berg Südwest,3
19,Reuterstraße,3
24,Schöneberg-Süd,3
26,Südliche Friedrichstadt,3


The first code takes the 10 biggest values in the way they're ordered, not matter what. The second one since keep = 'all' is selected that means according to the documentation: 'keep all the ties of the smallest item even if it means selecting more than n items.'. So that means that more as 10 items can be selected if there is a draw between number 10 and 11, which is the case here.

**8) Some users always look at the review comments to find if listings are clean and quiet. Let us define the cq-factor as the fraction of comments that mention both words `clean` and `quiet`, and return a data frame with the top 10 listings according to this cq-factor. Only return listings with at least 10 reviews. Tip: you can use [`str.contains`](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.contains.html); check argument `case`.**

In [311]:
combined = listings[listings['number_of_reviews'] > 10].merge(reviews, left_on='id', right_on='listing_id', how='inner')
per_listing = combined.groupby('listing_id').aggregate(total = ('listing_id', 'count'))
mask = reviews['comments'].str.contains('clean', case=False, na=False) & \
       reviews['comments'].str.contains('quiet', case=False, na=False)
per_listing['cq'] = reviews[mask].groupby('listing_id').aggregate(total = ('listing_id', 'count'))['total']
per_listing['factor'] = per_listing['cq'] / per_listing['total']
per_listing.sort_values('factor', ascending=False).head(10)

,total,cq,factor
listing_id,,,
739363,21,6.0,0.285714
240735,16,4.0,0.250000
268974,14,3.0,0.214286
782924,20,4.0,0.200000
109658,68,12.0,0.176471
231220,18,3.0,0.166667
617861,13,2.0,0.153846
28156,28,4.0,0.142857
237038,127,18.0,0.141732


**9) Let us use `reviews_per_month` as an indicator of the popularity of a listing, and the sum of `reviews_per_month` as the popularity of a set of listings. With these definitions, calculate the most popular room type per neighborhood group.**

In [247]:
aggregation = listings.groupby(['neighborhood_group', 'room_type'], as_index=False).aggregate(count = ('reviews_per_month', 'sum'))
df9 = aggregation.sort_values('count', ascending=False).drop_duplicates('neighborhood_group').sort_values('neighborhood_group')
display(df9)

,neighborhood_group,room_type,count
0,Charlottenburg-Wilm.,Entire home/apt,148.62
2,Friedrichshain-Kreuzberg,Entire home/apt,416.18
6,Lichtenberg,Private room,27.52
7,Marzahn - Hellersdorf,Entire home/apt,12.98
10,Mitte,Entire home/apt,499.17
14,Neukölln,Private room,224.90
16,Pankow,Entire home/apt,407.60
20,Reinickendorf,Private room,20.63
23,Spandau,Private room,7.09
24,Steglitz - Zehlendorf,Entire home/apt,28.43


**10) Create an _unambiguous_ and _nontrivial_ question, and its corresponding solution, as if you were writing the set of exercises for the lab. The question must cover at least 3 of the following aspects:**

- **Selecting columns or filtering rows**
- **Sorting rows**
- **Grouping and aggregation**
- **Creating/renaming/deleting columns or duplicates**
- **Joins**
- **Tidy data**
- **An open-ended conceptual question to explain some behavior**

**Please make it explicit which 3 of these aspects your question covers. You can use any of the datasets available on Brightspace.**